## 1. Project overview

This notebook separates **deterministic evidence construction** from **probabilistic narrative synthesis**.

Python retrieves structured data, resolves duplicate fiscal periods, constructs rolling TTM observations, calculates metrics, evaluates them against transparent rules, and records data-quality warnings. The LLM receives only a compact evidence payload. It is never asked to derive metrics from raw statements.

This is a lightweight alternative to RAG only for narrow questions whose evidence is already structured and numerically transformable. It does not replace retrieval over filings, transcripts, news, guidance, or other qualitative sources.

## 2. Install and import libraries

In [2]:
from __future__ import annotations

import json
import math
from dataclasses import dataclass
from datetime import date, datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field

## 3. Load secrets and define configuration

In [3]:
from google.colab import userdata

FMP_API_KEY = userdata.get("FMPAPIKEY")
OPENAI_API_KEY = userdata.get("OpenAIKey")

if not FMP_API_KEY:
    raise ValueError("Missing Colab secret: FMPAPIKEY")
if not OPENAI_API_KEY:
    raise ValueError("Missing Colab secret: OpenAIKey")

FMP_BASE_URL = "https://financialmodelingprep.com/stable"
OPENAI_MODEL = "gpt-5.4-mini"

BENCHMARK_SYMBOL = "SPY"
HISTORICAL_YEARS = 10
MIN_TTM_QUARTERS = 4
MIN_GROWTH_QUARTERS = 8
REQUEST_TIMEOUT_SECONDS = 30
TRADING_DAYS_PER_YEAR = 252
ANNUAL_RECONCILIATION_TOLERANCE = 0.08
BALANCE_SHEET_TOLERANCE = 0.03

EVALUATION_THRESHOLDS = {
    "history_percentile": {
        "strongly_above": 0.90,
        "above": 0.75,
        "below": 0.25,
        "strongly_below": 0.10,
    },
    "cash_conversion": {"strong": 1.00, "weak": 0.70},
    "net_debt_to_ebitda": {"strong": 1.0, "moderate": 2.5, "weak": 4.0},
    "interest_coverage": {"strong": 8.0, "adequate": 3.0, "weak": 1.5},
    "current_ratio": {"strong": 1.5, "adequate": 1.0},
    "relative_return_material": 0.10,
    "valuation_percentile": {"very_elevated": 0.90, "elevated": 0.75, "discounted": 0.25},
}

print(f"Configured OpenAI model: {OPENAI_MODEL}")

Configured OpenAI model: gpt-5.4-mini


## 4. FMP request functions

In [4]:
def fmp_get(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Any:
    request_params = dict(params or {})
    request_params["apikey"] = FMP_API_KEY
    response = requests.get(
        f"{FMP_BASE_URL}/{endpoint.lstrip('/')}",
        params=request_params,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise ValueError(payload["Error Message"])
    return payload


def get_company_profile(symbol: str) -> List[Dict[str, Any]]:
    return fmp_get("profile", {"symbol": symbol})


def get_income_statements(symbol: str, period: str = "quarter", limit: int = 40):
    return fmp_get("income-statement", {"symbol": symbol, "period": period, "limit": limit})


def get_balance_sheets(symbol: str, period: str = "quarter", limit: int = 40):
    return fmp_get("balance-sheet-statement", {"symbol": symbol, "period": period, "limit": limit})


def get_cash_flow_statements(symbol: str, period: str = "quarter", limit: int = 40):
    return fmp_get("cash-flow-statement", {"symbol": symbol, "period": period, "limit": limit})


def get_dividend_adjusted_prices(symbol: str, from_date: str, to_date: str):
    return fmp_get(
        "historical-price-eod/dividend-adjusted",
        {"symbol": symbol, "from": from_date, "to": to_date},
    )


def get_dividends(symbol: str, from_date: str, to_date: str):
    return fmp_get("dividends", {"symbol": symbol, "from": from_date, "to": to_date})


def get_benchmark_prices(symbol: str, from_date: str, to_date: str):
    return get_dividend_adjusted_prices(symbol, from_date, to_date)

## 5. Data normalization

In [5]:
INCOME_ALIASES = {
    "revenue": ["revenue"],
    "grossProfit": ["grossProfit"],
    "operatingIncome": ["operatingIncome"],
    "netIncome": ["netIncome"],
    "dilutedEPS": ["epsdiluted", "epsDiluted", "dilutedEPS"],
    "dilutedAverageShares": ["weightedAverageShsOutDil", "weightedAverageSharesDiluted"],
    "interestExpense": ["interestExpense", "interestExpenseNonOperating"],
    "incomeTaxExpense": ["incomeTaxExpense"],
    "incomeBeforeTax": ["incomeBeforeTax"],
    "ebitda": ["ebitda"],
}
BALANCE_ALIASES = {
    "cashAndShortTermInvestments": ["cashAndShortTermInvestments", "cashAndCashEquivalents"],
    "totalDebt": ["totalDebt"],
    "totalAssets": ["totalAssets"],
    "currentAssets": ["totalCurrentAssets", "currentAssets"],
    "currentLiabilities": ["totalCurrentLiabilities", "currentLiabilities"],
    "stockholdersEquity": ["totalStockholdersEquity", "stockholdersEquity", "totalEquity"],
    "totalLiabilities": ["totalLiabilities"],
}
CASH_FLOW_ALIASES = {
    "operatingCashFlow": ["operatingCashFlow", "netCashProvidedByOperatingActivities"],
    "capitalExpenditure": ["capitalExpenditure", "investmentsInPropertyPlantAndEquipment"],
    "freeCashFlow": ["freeCashFlow"],
    "stockBasedCompensation": ["stockBasedCompensation"],
    "dividendsPaidRaw": ["dividendsPaid", "commonDividendsPaid", "netDividendsPaid"],
    "shareRepurchasesRaw": ["commonStockRepurchased", "repurchasesOfCommonStock"],
    "changeInWorkingCapital": ["changeInWorkingCapital"],
    "netDebtIssuance": ["netDebtIssuance", "debtRepayment", "issuanceRepaymentOfDebtSecurities"],
}


def first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    return next((c for c in candidates if c in df.columns), None)


def coalesce_aliases(df: pd.DataFrame, aliases: Dict[str, List[str]]) -> pd.DataFrame:
    out = df.copy()
    for canonical, candidates in aliases.items():
        series = pd.Series(np.nan, index=out.index, dtype="float64")
        for candidate in candidates:
            if candidate in out.columns:
                series = series.fillna(pd.to_numeric(out[candidate], errors="coerce"))
        out[canonical] = series
    return out


def normalize_statement(records, aliases, name) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    if not records:
        return pd.DataFrame(), {"statement": name, "input_rows": 0, "output_rows": 0, "duplicate_periods_resolved": 0}
    df = pd.DataFrame(records)
    if "date" not in df.columns:
        raise ValueError(f"{name} has no date field.")
    for c in ["date", "filingDate", "acceptedDate"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    df = df.rename(columns={"date": "fiscal_period_end"})
    df = coalesce_aliases(df, aliases)
    for c in ["filingDate", "acceptedDate"]:
        if c not in df.columns:
            df[c] = pd.NaT
    if "reportedCurrency" not in df.columns:
        df["reportedCurrency"] = None

    input_rows = len(df)
    df = (
        df.sort_values(["fiscal_period_end", "acceptedDate", "filingDate"], na_position="first")
        .drop_duplicates("fiscal_period_end", keep="last")
        .sort_values("fiscal_period_end")
        .reset_index(drop=True)
    )
    return df, {
        "statement": name,
        "input_rows": input_rows,
        "output_rows": len(df),
        "duplicate_periods_resolved": input_rows - len(df),
    }


def normalize_prices(records) -> pd.DataFrame:
    if not records:
        return pd.DataFrame(columns=["date", "price"])
    df = pd.DataFrame(records)
    price_col = first_existing(df, ["adjClose", "adjustedClose", "close"])
    if "date" not in df.columns or not price_col:
        raise ValueError("Price response lacks date or adjusted price.")
    out = pd.DataFrame({
        "date": pd.to_datetime(df["date"], errors="coerce"),
        "price": pd.to_numeric(df[price_col], errors="coerce"),
    })
    return out.dropna().drop_duplicates("date", keep="last").sort_values("date").reset_index(drop=True)


def normalize_dividends(records) -> pd.DataFrame:
    if not records:
        return pd.DataFrame(columns=["date", "dividend"])
    df = pd.DataFrame(records)
    date_col = first_existing(df, ["date", "paymentDate", "recordDate"])
    amount_col = first_existing(df, ["adjDividend", "dividend", "amount"])
    if not date_col or not amount_col:
        return pd.DataFrame(columns=["date", "dividend"])
    out = pd.DataFrame({
        "date": pd.to_datetime(df[date_col], errors="coerce"),
        "dividend": pd.to_numeric(df[amount_col], errors="coerce"),
    })
    return out.dropna().drop_duplicates().sort_values("date").reset_index(drop=True)


def align_quarterly_statements(income, balance, cash_flow) -> pd.DataFrame:
    income_cols = ["fiscal_period_end", "filingDate", "acceptedDate", "reportedCurrency", *INCOME_ALIASES]
    balance_cols = ["fiscal_period_end", *BALANCE_ALIASES]
    cash_cols = ["fiscal_period_end", *CASH_FLOW_ALIASES]
    aligned = income[[c for c in income_cols if c in income]].merge(
        balance[[c for c in balance_cols if c in balance]], on="fiscal_period_end", how="outer"
    ).merge(
        cash_flow[[c for c in cash_cols if c in cash_flow]], on="fiscal_period_end", how="outer"
    )
    aligned["capitalExpenditureOutflow"] = aligned["capitalExpenditure"].abs()
    aligned["dividendsPaid"] = aligned["dividendsPaidRaw"].abs()
    aligned["shareRepurchases"] = aligned["shareRepurchasesRaw"].where(aligned["shareRepurchasesRaw"] <= 0).abs()
    aligned["freeCashFlow"] = aligned["freeCashFlow"].fillna(
        aligned["operatingCashFlow"] - aligned["capitalExpenditureOutflow"]
    )
    return aligned.sort_values("fiscal_period_end").reset_index(drop=True)

## 6. TTM calculation engine

In [6]:
FLOW_AGGREGATIONS = {
    "revenue": "sum", "grossProfit": "sum", "operatingIncome": "sum", "netIncome": "sum",
    "dilutedEPS": "sum", "operatingCashFlow": "sum", "capitalExpenditureOutflow": "sum",
    "freeCashFlow": "sum", "stockBasedCompensation": "sum", "dividendsPaid": "sum",
    "shareRepurchases": "sum", "changeInWorkingCapital": "sum", "netDebtIssuance": "sum",
    "interestExpense": "sum", "incomeTaxExpense": "sum", "incomeBeforeTax": "sum",
    "ebitda": "sum", "dilutedAverageShares": "mean",
}


def safe_divide(numerator, denominator, require_positive_denominator=False):
    if numerator is None or denominator is None or pd.isna(numerator) or pd.isna(denominator) or denominator == 0:
        return None
    if require_positive_denominator and denominator <= 0:
        return None
    return float(numerator / denominator)


def meaningful_growth(current, prior):
    if current is None or prior is None or pd.isna(current) or pd.isna(prior):
        return None, "insufficient_data"
    if prior <= 0:
        return None, "not_meaningful"
    return float(current / prior - 1), "valid"


def percentile_rank(values, current):
    clean = np.array([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if current is None or not np.isfinite(current) or clean.size == 0:
        return None
    return float(np.mean(clean <= current))


def rolling_ttm(quarterly: pd.DataFrame, column: str, aggregation: Optional[str] = None):
    if column not in quarterly:
        return pd.DataFrame(columns=["fiscal_period_end", "value"])
    aggregation = aggregation or FLOW_AGGREGATIONS.get(column, "sum")
    series = pd.to_numeric(quarterly[column], errors="coerce")
    values = series.rolling(4, min_periods=4).mean() if aggregation == "mean" else series.rolling(4, min_periods=4).sum()
    return pd.DataFrame({"fiscal_period_end": quarterly["fiscal_period_end"], "value": values}).dropna().reset_index(drop=True)


def rolling_ratio(quarterly, numerator, denominator="revenue"):
    n = rolling_ttm(quarterly, numerator).rename(columns={"value": "numerator"})
    d = rolling_ttm(quarterly, denominator).rename(columns={"value": "denominator"})
    out = n.merge(d, on="fiscal_period_end")
    out["value"] = [safe_divide(a, b, True) for a, b in zip(out["numerator"], out["denominator"])]
    return out.dropna(subset=["value"])[["fiscal_period_end", "value"]]


def summarize_history(history: pd.DataFrame):
    if history.empty:
        return {
            "current": None, "prior": None, "current_yoy_growth": None, "growth_status": "insufficient_data",
            "previous_yoy_growth": None, "growth_acceleration_pp": None, "historical_median_growth": None,
            "historical_growth_q25": None, "historical_growth_q75": None,
            "current_growth_percentile": None, "observations": 0,
        }
    values = history["value"].astype(float).reset_index(drop=True)
    current = float(values.iloc[-1])
    prior = float(values.iloc[-5]) if len(values) >= 5 else None
    current_growth, status = meaningful_growth(current, prior)
    growths = []
    for i in range(4, len(values)):
        g, s = meaningful_growth(float(values.iloc[i]), float(values.iloc[i - 4]))
        growths.append(g if s == "valid" else None)
    valid = [g for g in growths if g is not None]
    previous = valid[-2] if len(valid) >= 2 else None
    return {
        "current": current,
        "prior": prior,
        "current_yoy_growth": current_growth,
        "growth_status": status,
        "previous_yoy_growth": previous,
        "growth_acceleration_pp": (current_growth - previous) * 100 if current_growth is not None and previous is not None else None,
        "historical_median_growth": float(np.median(valid)) if valid else None,
        "historical_growth_q25": float(np.percentile(valid, 25)) if valid else None,
        "historical_growth_q75": float(np.percentile(valid, 75)) if valid else None,
        "current_growth_percentile": percentile_rank(valid, current_growth),
        "observations": len(valid),
    }


def ttm_value(q, column):
    h = rolling_ttm(q, column)
    return float(h["value"].iloc[-1]) if not h.empty else None


def latest_snapshot(q, column):
    if column not in q:
        return None
    s = pd.to_numeric(q[column], errors="coerce").dropna()
    return float(s.iloc[-1]) if not s.empty else None


def prior_year_snapshot(q, column):
    if column not in q or len(q) < 5:
        return None
    value = pd.to_numeric(q[column], errors="coerce").iloc[-5]
    return float(value) if pd.notna(value) else None

## 7. Financial metric modules

In [7]:
def serializable(value):
    if isinstance(value, dict):
        return {str(k): serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [serializable(v) for v in value]
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return value.isoformat()
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value


def history_assessment(percentile, higher_is_better=True):
    if percentile is None:
        return "insufficient_data"
    t = EVALUATION_THRESHOLDS["history_percentile"]
    if higher_is_better:
        if percentile >= t["strongly_above"]: return "strongly_above_history"
        if percentile >= t["above"]: return "above_history"
        if percentile <= t["strongly_below"]: return "strongly_below_history"
        if percentile <= t["below"]: return "below_history"
    else:
        if percentile >= t["strongly_above"]: return "strongly_below_history"
        if percentile >= t["above"]: return "below_history"
        if percentile <= t["strongly_below"]: return "strongly_above_history"
        if percentile <= t["below"]: return "above_history"
    return "in_line_with_history"


def evidence_item(evidence_id, metric, value, unit, as_of=None, comparison=None, assessment="insufficient_data", formula=None, notes=None):
    return serializable({
        "evidence_id": evidence_id, "metric": metric, "value": value, "unit": unit,
        "as_of": as_of, "comparison": comparison or {}, "assessment": assessment,
        "formula": formula, "notes": notes or [],
    })


def build_growth_metrics(q, as_of):
    specs = [
        ("G01", "revenue_growth", "revenue"), ("G02", "gross_profit_growth", "grossProfit"),
        ("G03", "operating_income_growth", "operatingIncome"), ("G04", "net_income_growth", "netIncome"),
        ("G05", "diluted_eps_growth", "dilutedEPS"), ("G06", "operating_cash_flow_growth", "operatingCashFlow"),
        ("G07", "free_cash_flow_growth", "freeCashFlow"), ("G08", "diluted_share_count_growth", "dilutedAverageShares"),
    ]
    out = {}
    for eid, name, col in specs:
        summary = summarize_history(rolling_ttm(q, col))
        assessment = summary["growth_status"] if summary["growth_status"] != "valid" else history_assessment(summary["current_growth_percentile"])
        out[name] = evidence_item(
            eid, name, summary["current_yoy_growth"], "ratio", as_of,
            {
                "current_ttm": summary["current"], "prior_ttm": summary["prior"],
                "previous_ttm_yoy_growth": summary["previous_yoy_growth"],
                "growth_acceleration_pp": summary["growth_acceleration_pp"],
                "historical_median_growth": summary["historical_median_growth"],
                "historical_q25_growth": summary["historical_growth_q25"],
                "historical_q75_growth": summary["historical_growth_q75"],
                "historical_percentile": summary["current_growth_percentile"],
                "observations": summary["observations"],
            },
            assessment, "current TTM / prior TTM - 1"
        )
    return out


def margin_summary(q, numerator):
    h = rolling_ratio(q, numerator)
    if h.empty:
        return {}
    values = h["value"].tolist()
    current = values[-1]
    prior = values[-5] if len(values) >= 5 else None
    history = values[:-1] if len(values) > 1 else values
    return {
        "current": current, "prior": prior, "change_pp": (current - prior) * 100 if prior is not None else None,
        "historical_median": float(np.median(history)) if history else None,
        "historical_q25": float(np.percentile(history, 25)) if history else None,
        "historical_q75": float(np.percentile(history, 75)) if history else None,
        "historical_percentile": percentile_rank(history, current), "observations": len(history),
    }


def build_profitability_metrics(q, as_of):
    specs = [("P01", "gross_margin", "grossProfit"), ("P02", "operating_margin", "operatingIncome"),
             ("P03", "net_margin", "netIncome"), ("P04", "operating_cash_flow_margin", "operatingCashFlow"),
             ("P05", "free_cash_flow_margin", "freeCashFlow")]
    out = {}
    for eid, name, numerator in specs:
        s = margin_summary(q, numerator)
        out[name] = evidence_item(
            eid, name, s.get("current"), "ratio", as_of,
            {
                "prior_ttm_margin": s.get("prior"), "change_pp": s.get("change_pp"),
                "historical_median": s.get("historical_median"), "historical_q25": s.get("historical_q25"),
                "historical_q75": s.get("historical_q75"), "historical_percentile": s.get("historical_percentile"),
                "observations": s.get("observations", 0),
            },
            history_assessment(s.get("historical_percentile")),
            f"TTM {numerator} / TTM revenue",
        )
    return out


def build_cash_flow_metrics(q, as_of):
    values = {
        "operating_cash_flow_to_net_income": (ttm_value(q, "operatingCashFlow"), ttm_value(q, "netIncome")),
        "free_cash_flow_to_net_income": (ttm_value(q, "freeCashFlow"), ttm_value(q, "netIncome")),
        "capital_expenditures_to_revenue": (ttm_value(q, "capitalExpenditureOutflow"), ttm_value(q, "revenue")),
        "stock_based_compensation_to_revenue": (ttm_value(q, "stockBasedCompensation"), ttm_value(q, "revenue")),
        "change_in_working_capital_to_revenue": (ttm_value(q, "changeInWorkingCapital"), ttm_value(q, "revenue")),
    }
    ids = {"operating_cash_flow_to_net_income": "CF01", "free_cash_flow_to_net_income": "CF02",
           "capital_expenditures_to_revenue": "CF03", "stock_based_compensation_to_revenue": "CF04",
           "change_in_working_capital_to_revenue": "CF05"}
    out = {}
    for name, (num, den) in values.items():
        value = safe_divide(num, den, True)
        if value is None:
            assessment = "not_meaningful" if den is not None and den <= 0 else "insufficient_data"
        elif name in {"operating_cash_flow_to_net_income", "free_cash_flow_to_net_income"}:
            assessment = "above_history" if value >= 1 else "below_history" if value < 0.7 else "in_line_with_history"
        else:
            assessment = "in_line_with_history"
        out[name] = evidence_item(ids[name], name, value, "ratio", as_of, assessment=assessment)
    return out


def build_balance_sheet_metrics(q, as_of):
    cash, debt = latest_snapshot(q, "cashAndShortTermInvestments"), latest_snapshot(q, "totalDebt")
    net_debt = debt - cash if debt is not None and cash is not None else None
    current_ratio = safe_divide(latest_snapshot(q, "currentAssets"), latest_snapshot(q, "currentLiabilities"), True)
    nd_ebitda = safe_divide(net_debt, ttm_value(q, "ebitda"), True)
    interest = ttm_value(q, "interestExpense")
    interest_coverage = safe_divide(ttm_value(q, "operatingIncome"), abs(interest) if interest is not None else None, True)
    debt_growth, debt_status = meaningful_growth(debt, prior_year_snapshot(q, "totalDebt"))
    cash_growth, cash_status = meaningful_growth(cash, prior_year_snapshot(q, "cashAndShortTermInvestments"))

    leverage_assessment = "insufficient_data"
    if nd_ebitda is not None:
        t = EVALUATION_THRESHOLDS["net_debt_to_ebitda"]
        leverage_assessment = ("strongly_above_history" if nd_ebitda <= t["strong"] else
                               "above_history" if nd_ebitda <= t["moderate"] else
                               "strongly_below_history" if nd_ebitda >= t["weak"] else "below_history")
    return {
        "cash_and_short_term_investments": evidence_item("BS01", "cash_and_short_term_investments", cash, "currency", as_of, assessment="in_line_with_history"),
        "total_debt": evidence_item("BS02", "total_debt", debt, "currency", as_of, assessment="in_line_with_history"),
        "net_debt": evidence_item("BS03", "net_debt", net_debt, "currency", as_of, assessment="above_history" if net_debt is not None and net_debt < 0 else "in_line_with_history", formula="total debt - cash"),
        "current_ratio": evidence_item("BS04", "current_ratio", current_ratio, "ratio", as_of, assessment="above_history" if current_ratio is not None and current_ratio >= 1.5 else "below_history" if current_ratio is not None and current_ratio < 1 else "in_line_with_history" if current_ratio is not None else "insufficient_data"),
        "net_debt_to_ebitda": evidence_item("BS05", "net_debt_to_ebitda", nd_ebitda, "multiple", as_of, assessment=leverage_assessment),
        "interest_coverage": evidence_item("BS06", "interest_coverage", interest_coverage, "multiple", as_of, assessment="above_history" if interest_coverage is not None and interest_coverage >= 8 else "below_history" if interest_coverage is not None and interest_coverage < 3 else "in_line_with_history" if interest_coverage is not None else "insufficient_data"),
        "debt_growth_yoy": evidence_item("BS07", "debt_growth_yoy", debt_growth, "ratio", as_of, assessment=debt_status if debt_status != "valid" else "below_history" if debt_growth > 0 else "above_history"),
        "cash_growth_yoy": evidence_item("BS08", "cash_growth_yoy", cash_growth, "ratio", as_of, assessment=cash_status if cash_status != "valid" else "above_history" if cash_growth > 0 else "below_history"),
    }


def average_balance(q, column):
    if column not in q or len(q) < 5:
        return None
    end, begin = pd.to_numeric(q[column], errors="coerce").iloc[-1], pd.to_numeric(q[column], errors="coerce").iloc[-5]
    return float((end + begin) / 2) if pd.notna(end) and pd.notna(begin) else None


def build_capital_efficiency_metrics(q, as_of):
    ni, op = ttm_value(q, "netIncome"), ttm_value(q, "operatingIncome")
    tax_rate = safe_divide(ttm_value(q, "incomeTaxExpense"), ttm_value(q, "incomeBeforeTax"), True)
    if tax_rate is None or not 0 <= tax_rate <= 0.5:
        tax_rate = 0.21
    avg_assets, avg_equity = average_balance(q, "totalAssets"), average_balance(q, "stockholdersEquity")
    end_ic = None
    begin_ic = None
    values_end = [latest_snapshot(q, c) for c in ["totalDebt", "stockholdersEquity", "cashAndShortTermInvestments"]]
    values_begin = [prior_year_snapshot(q, c) for c in ["totalDebt", "stockholdersEquity", "cashAndShortTermInvestments"]]
    if None not in values_end: end_ic = values_end[0] + values_end[1] - values_end[2]
    if None not in values_begin: begin_ic = values_begin[0] + values_begin[1] - values_begin[2]
    avg_ic = (end_ic + begin_ic) / 2 if end_ic is not None and begin_ic is not None else None
    roa, roe = safe_divide(ni, avg_assets, True), safe_divide(ni, avg_equity, True)
    roic = safe_divide(op * (1 - tax_rate) if op is not None else None, avg_ic, True)
    return {
        "return_on_assets": evidence_item("CE01", "return_on_assets", roa, "ratio", as_of, assessment="in_line_with_history" if roa is not None else "insufficient_data"),
        "return_on_equity": evidence_item("CE02", "return_on_equity", roe, "ratio", as_of, assessment="not_meaningful" if avg_equity is not None and avg_equity <= 0 else "in_line_with_history" if roe is not None else "insufficient_data", notes=["Negative equity can make ROE misleading."] if avg_equity is not None and avg_equity <= 0 else []),
        "return_on_invested_capital_approx": evidence_item("CE03", "return_on_invested_capital_approx", roic, "ratio", as_of, assessment="not_meaningful" if avg_ic is not None and avg_ic <= 0 else "in_line_with_history" if roic is not None else "insufficient_data", formula="NOPAT / average(debt + equity - cash)", notes=["Uses a 21% fallback tax rate when reported tax rate is invalid."]),
    }


def build_capital_allocation_metrics(q, dividends, as_of):
    divs, buybacks, debt_issuance, fcf = [ttm_value(q, c) for c in ["dividendsPaid", "shareRepurchases", "netDebtIssuance", "freeCashFlow"]]
    share_summary = summarize_history(rolling_ttm(q, "dilutedAverageShares"))
    repurchases_to_fcf = safe_divide(buybacks, fcf, True)
    total_returned = divs + buybacks if divs is not None and buybacks is not None else None
    total_to_fcf = safe_divide(total_returned, fcf, True)
    dividend_growth, dividend_status = None, "insufficient_data"
    if not dividends.empty:
        annual = dividends.assign(year=dividends["date"].dt.year).groupby("year", as_index=False)["dividend"].sum()
        if len(annual) >= 2:
            dividend_growth, dividend_status = meaningful_growth(float(annual["dividend"].iloc[-1]), float(annual["dividend"].iloc[-2]))
    return {
        "dividends_paid": evidence_item("CA01", "dividends_paid", divs, "currency", as_of, assessment="in_line_with_history", notes=["Positive outflow magnitude."]),
        "share_repurchases": evidence_item("CA02", "share_repurchases", buybacks, "currency", as_of, assessment="in_line_with_history", notes=["Positive outflow magnitude."]),
        "net_debt_issuance": evidence_item("CA03", "net_debt_issuance", debt_issuance, "currency", as_of, assessment="below_history" if debt_issuance is not None and debt_issuance > 0 else "above_history" if debt_issuance is not None and debt_issuance < 0 else "insufficient_data"),
        "diluted_share_count_change": evidence_item("CA04", "diluted_share_count_change", share_summary["current_yoy_growth"], "ratio", as_of, assessment=share_summary["growth_status"] if share_summary["growth_status"] != "valid" else "above_history" if share_summary["current_yoy_growth"] < 0 else "below_history"),
        "repurchases_as_pct_of_fcf": evidence_item("CA05", "repurchases_as_pct_of_fcf", repurchases_to_fcf, "ratio", as_of, assessment="in_line_with_history" if repurchases_to_fcf is not None else "not_meaningful"),
        "total_capital_returned_as_pct_of_fcf": evidence_item("CA06", "total_capital_returned_as_pct_of_fcf", total_to_fcf, "ratio", as_of, assessment="below_history" if total_to_fcf is not None and total_to_fcf > 1.25 else "in_line_with_history" if total_to_fcf is not None else "not_meaningful"),
        "dividend_growth": evidence_item("CA07", "dividend_growth", dividend_growth, "ratio", dividends["date"].max() if not dividends.empty else as_of, assessment=dividend_status if dividend_status != "valid" else "above_history" if dividend_growth > 0 else "below_history"),
    }

In [8]:
def price_on_or_before(prices, target):
    eligible = prices.loc[prices["date"] <= target, "price"]
    return float(eligible.iloc[-1]) if not eligible.empty else None


def period_return(prices, days):
    if prices.empty: return None
    end_date, end_price = prices["date"].iloc[-1], float(prices["price"].iloc[-1])
    start_price = price_on_or_before(prices, end_date - pd.Timedelta(days=days))
    return end_price / start_price - 1 if start_price and start_price > 0 else None


def annualized_return(prices, years):
    if prices.empty: return None
    end_date, end_price = prices["date"].iloc[-1], float(prices["price"].iloc[-1])
    start_price = price_on_or_before(prices, end_date - pd.DateOffset(years=years))
    return (end_price / start_price) ** (1 / years) - 1 if start_price and start_price > 0 else None


def market_statistics(prices, benchmark):
    if prices.empty: return {}
    cutoff = prices["date"].iloc[-1] - pd.DateOffset(years=1)
    daily = prices.set_index("date")["price"].pct_change().dropna()
    daily_1y = daily.loc[daily.index >= cutoff]
    drawdown = prices["price"] / prices["price"].cummax() - 1
    trailing = prices.loc[prices["date"] >= cutoff, "price"]
    stock_1y, bench_1y = period_return(prices, 365), period_return(benchmark, 365)
    aligned = prices.rename(columns={"price": "stock"}).merge(
        benchmark.rename(columns={"price": "benchmark"}), on="date"
    ).set_index("date").pct_change().dropna()
    aligned = aligned.loc[aligned.index >= cutoff]
    beta = aligned["stock"].cov(aligned["benchmark"]) / aligned["benchmark"].var() if len(aligned) >= 30 and aligned["benchmark"].var() > 0 else None
    return {
        "return_1m": period_return(prices, 30), "return_3m": period_return(prices, 91),
        "return_6m": period_return(prices, 182), "return_1y": stock_1y,
        "return_3y_annualized": annualized_return(prices, 3),
        "volatility_1y_annualized": float(daily_1y.std(ddof=1) * np.sqrt(252)) if len(daily_1y) >= 30 else None,
        "maximum_drawdown": float(drawdown.min()) if not drawdown.empty else None,
        "distance_from_52_week_high": float(prices["price"].iloc[-1] / trailing.max() - 1) if not trailing.empty else None,
        "benchmark_return_1y": bench_1y,
        "relative_return_1y": stock_1y - bench_1y if stock_1y is not None and bench_1y is not None else None,
        "beta_1y": float(beta) if beta is not None else None,
    }


def historical_valuation_series(q, prices):
    required = [rolling_ttm(q, c).rename(columns={"value": c}) for c in ["revenue", "netIncome", "freeCashFlow", "ebitda", "dilutedAverageShares"]]
    merged = required[0]
    for frame in required[1:]:
        merged = merged.merge(frame, on="fiscal_period_end")
    merged = merged.merge(q[["fiscal_period_end", "totalDebt", "cashAndShortTermInvestments"]], on="fiscal_period_end", how="left")
    series = {"pe": [], "ev_revenue": [], "ev_ebitda": [], "p_fcf": []}
    for row in merged.itertuples(index=False):
        px = price_on_or_before(prices, row.fiscal_period_end)
        if px is None or row.dilutedAverageShares <= 0: continue
        market_cap = px * row.dilutedAverageShares
        ev = market_cap + row.totalDebt - row.cashAndShortTermInvestments
        if row.netIncome > 0: series["pe"].append(market_cap / row.netIncome)
        if row.revenue > 0: series["ev_revenue"].append(ev / row.revenue)
        if row.ebitda > 0: series["ev_ebitda"].append(ev / row.ebitda)
        if row.freeCashFlow > 0: series["p_fcf"].append(market_cap / row.freeCashFlow)
    return series


def valuation_assessment(current, history):
    p = percentile_rank(history, current)
    if p is None: return "insufficient_data", None
    t = EVALUATION_THRESHOLDS["valuation_percentile"]
    if p >= t["very_elevated"]: return "strongly_below_history", p
    if p >= t["elevated"]: return "below_history", p
    if p <= t["discounted"]: return "above_history", p
    return "in_line_with_history", p


def build_valuation_metrics(profile, q, prices, as_of):
    latest_price = float(prices["price"].iloc[-1]) if not prices.empty else None
    profile_mcap = pd.to_numeric(profile.get("marketCap"), errors="coerce")
    shares = ttm_value(q, "dilutedAverageShares")
    market_cap = float(profile_mcap) if pd.notna(profile_mcap) and profile_mcap > 0 else latest_price * shares if latest_price and shares else None
    revenue, ni, ebitda, fcf = [ttm_value(q, c) for c in ["revenue", "netIncome", "ebitda", "freeCashFlow"]]
    debt, cash = latest_snapshot(q, "totalDebt"), latest_snapshot(q, "cashAndShortTermInvestments")
    ev = market_cap + debt - cash if None not in [market_cap, debt, cash] else None
    current = {
        "price_to_earnings": safe_divide(market_cap, ni, True),
        "enterprise_value_to_revenue": safe_divide(ev, revenue, True),
        "enterprise_value_to_ebitda": safe_divide(ev, ebitda, True),
        "price_to_free_cash_flow": safe_divide(market_cap, fcf, True),
        "earnings_yield": safe_divide(ni, market_cap, True),
        "free_cash_flow_yield": safe_divide(fcf, market_cap, True),
    }
    hist = historical_valuation_series(q, prices)
    hmap = {"price_to_earnings": "pe", "enterprise_value_to_revenue": "ev_revenue", "enterprise_value_to_ebitda": "ev_ebitda", "price_to_free_cash_flow": "p_fcf"}
    ids = dict(zip(current, ["V01", "V02", "V03", "V04", "V05", "V06"]))
    out = {}
    for name, value in current.items():
        if name in hmap:
            assessment, p = valuation_assessment(value, hist[hmap[name]])
            comparison = {"historical_percentile": p, "historical_observations": len(hist[hmap[name]]), "historical_median": float(np.median(hist[hmap[name]])) if hist[hmap[name]] else None}
        else:
            assessment = "above_history" if value is not None and value > 0.05 else "in_line_with_history" if value is not None else "not_meaningful"
            comparison = {}
        out[name] = evidence_item(ids[name], name, value, "ratio" if "yield" in name else "multiple", as_of, comparison, assessment, notes=["Multiple omitted when denominator is zero or negative."] if value is None else [])
    return out


def build_market_metrics(prices, benchmark, as_of):
    stats = market_statistics(prices, benchmark)
    specs = [("M01", "return_1m"), ("M02", "return_3m"), ("M03", "return_6m"), ("M04", "return_1y"),
             ("M05", "return_3y_annualized"), ("M06", "volatility_1y_annualized"), ("M07", "maximum_drawdown"),
             ("M08", "distance_from_52_week_high"), ("M09", "relative_return_1y"), ("M10", "beta_1y")]
    out = {}
    for eid, name in specs:
        value = stats.get(name)
        if name == "relative_return_1y" and value is not None:
            assessment = "above_history" if value > 0.10 else "below_history" if value < -0.10 else "in_line_with_history"
        elif name in {"maximum_drawdown", "distance_from_52_week_high"}:
            assessment = "below_history" if value is not None and value < -0.30 else "in_line_with_history" if value is not None else "insufficient_data"
        else:
            assessment = "in_line_with_history" if value is not None else "insufficient_data"
        comparison = {"stock_return_1y": stats.get("return_1y"), "benchmark_return_1y": stats.get("benchmark_return_1y"), "benchmark_symbol": BENCHMARK_SYMBOL} if name == "relative_return_1y" else {}
        out[name] = evidence_item(eid, name, value, "beta" if name == "beta_1y" else "ratio", as_of, comparison, assessment)
    return out

## 8. Deterministic evaluation layer

In [9]:
ASSESSMENT_SEVERITY = {
    "strongly_above_history": "high_positive", "above_history": "moderate_positive",
    "in_line_with_history": "neutral", "below_history": "moderate_negative",
    "strongly_below_history": "high_negative", "not_meaningful": "not_applicable",
    "insufficient_data": "unknown",
}


def direction_from_assessment(a):
    if a in {"strongly_above_history", "above_history"}: return "positive"
    if a in {"strongly_below_history", "below_history"}: return "negative"
    if a == "in_line_with_history": return "neutral"
    return "unknown"


def deterministic_explanation(item):
    metric = item["metric"].replace("_", " ").title()
    assessment = item["assessment"]
    if assessment == "insufficient_data": return f"{metric} could not be evaluated because required data were unavailable."
    if assessment == "not_meaningful": return f"{metric} is not meaningful under the available denominator or accounting condition."
    comparison = item.get("comparison", {})
    med = comparison.get("historical_median_growth", comparison.get("historical_median"))
    if med is not None and item.get("value") is not None:
        return f"{metric} is {'above' if item['value'] > med else 'below'} its historical median and is classified as {assessment}."
    return f"{metric} is classified as {assessment} under the configured deterministic rule."


def build_signals(sections):
    signals = []
    for section, metrics in sections.items():
        for item in metrics.values():
            assessment = item["assessment"]
            signals.append({
                "metric_id": item["evidence_id"],
                "rule_description": "Use historical percentile thresholds when available; otherwise use the metric-specific configured threshold.",
                "raw_result": item.get("value"),
                "historical_comparison": item.get("comparison", {}),
                "direction": direction_from_assessment(assessment),
                "severity": ASSESSMENT_SEVERITY[assessment],
                "assessment": assessment,
                "evidence_ids": [item["evidence_id"]],
                "explanation": deterministic_explanation(item),
            })
    positive_growth = [x for x in sections["growth"].values() if x["assessment"] in {"above_history", "strongly_above_history"}]
    relative = sections["market_performance"].get("relative_return_1y")
    if len(positive_growth) >= 2 and relative and relative.get("value") is not None and relative["value"] < 0:
        signals.append({
            "metric_id": "X01", "rule_description": "Flag improving fundamentals with negative one-year relative performance.",
            "raw_result": {"positive_growth_signals": len(positive_growth), "relative_return_1y": relative["value"]},
            "historical_comparison": {}, "direction": "mixed", "severity": "moderate",
            "assessment": "conflicting_evidence",
            "evidence_ids": [x["evidence_id"] for x in positive_growth[:3]] + [relative["evidence_id"]],
            "explanation": "One-year relative performance is negative despite multiple above-history growth signals.",
        })
    return serializable(signals)

## 9. Data-quality checks

In [10]:
REQUIRED_QUARTERLY_FIELDS = ["revenue", "netIncome", "operatingCashFlow", "freeCashFlow", "totalAssets", "stockholdersEquity"]


def periods_appear_consecutive(dates):
    clean = pd.Series(pd.to_datetime(dates, errors="coerce")).dropna().sort_values().tail(8)
    return len(clean) >= 4 and clean.diff().dt.days.dropna().between(65, 125).all()


def annual_reconciliation(q, annual_income, annual_cash):
    results = {}
    for metric, annual in [("revenue", annual_income), ("netIncome", annual_income), ("operatingCashFlow", annual_cash), ("freeCashFlow", annual_cash)]:
        if annual.empty or metric not in annual:
            results[metric] = {"status": "not_available"}; continue
        clean = annual.dropna(subset=["fiscal_period_end", metric])
        if clean.empty:
            results[metric] = {"status": "not_available"}; continue
        row = clean.iloc[-1]
        year = row["fiscal_period_end"].year
        quarters = pd.to_numeric(q.loc[q["fiscal_period_end"].dt.year == year, metric], errors="coerce").dropna()
        if len(quarters) != 4:
            results[metric] = {"status": "insufficient_matching_quarters", "fiscal_year": year, "matching_quarters": len(quarters)}; continue
        annual_value, quarter_sum = float(row[metric]), float(quarters.sum())
        diff = abs(quarter_sum - annual_value) / max(abs(annual_value), 1)
        results[metric] = {"status": "pass" if diff <= ANNUAL_RECONCILIATION_TOLERANCE else "warning", "fiscal_year": year, "annual_value": annual_value, "quarter_sum": quarter_sum, "relative_difference": diff}
    return serializable(results)


def balance_reconciliation(q):
    assets, liabilities, equity = [latest_snapshot(q, c) for c in ["totalAssets", "totalLiabilities", "stockholdersEquity"]]
    if None in [assets, liabilities, equity]: return {"status": "not_available"}
    diff = abs(assets - liabilities - equity) / max(abs(assets), 1)
    return {"status": "pass" if diff <= BALANCE_SHEET_TOLERANCE else "warning", "assets": assets, "liabilities_plus_equity": liabilities + equity, "relative_difference": diff}


def build_data_quality(q, metadata, prices, annual_income, annual_cash):
    warnings = []
    if len(q) < 4: warnings.append("Fewer than four aligned quarters are available for TTM calculations.")
    if len(q) < 8: warnings.append("Fewer than eight aligned quarters are available for TTM growth.")
    consecutive = periods_appear_consecutive(q["fiscal_period_end"])
    if not consecutive: warnings.append("Latest fiscal periods do not appear consistently quarterly.")
    duplicates = sum(m["duplicate_periods_resolved"] for m in metadata)
    if duplicates: warnings.append(f"{duplicates} duplicate/restated periods were resolved using the latest accepted filing.")
    missing = [c for c in REQUIRED_QUARTERLY_FIELDS if c not in q or pd.to_numeric(q[c], errors="coerce").tail(4).isna().any()]
    if missing: warnings.append("Required recent fields are missing: " + ", ".join(missing))
    unique, sorted_dates = (bool(prices["date"].is_unique), bool(prices["date"].is_monotonic_increasing)) if not prices.empty else (False, False)
    if not unique or not sorted_dates: warnings.append("Market-price dates are not unique and sorted.")
    annual = annual_reconciliation(q, annual_income, annual_cash)
    if any(v.get("status") == "warning" for v in annual.values()): warnings.append("At least one annual flow does not approximately reconcile to quarterly values.")
    balance = balance_reconciliation(q)
    if balance.get("status") == "warning": warnings.append("Assets do not approximately equal liabilities plus equity.")
    status = "poor" if len(q) < 4 or len(missing) >= 3 else "warning" if warnings else "good"
    return serializable({
        "overall_status": status, "number_of_aligned_quarters": len(q),
        "latest_filing_date": q["filingDate"].dropna().max() if "filingDate" in q else None,
        "latest_market_date": prices["date"].max() if not prices.empty else None,
        "latest_periods_consecutive": consecutive, "duplicate_periods_resolved": duplicates,
        "annual_reconciliation": annual, "balance_sheet_reconciliation": balance,
        "market_price_dates_unique": unique, "market_price_dates_sorted": sorted_dates,
        "missing_metrics": missing, "warnings": warnings,
    })

## 10. Canonical evidence summary

In [11]:
def normalize_symbol(symbol):
    symbol = str(symbol).strip().upper()
    if not symbol or len(symbol) > 15:
        raise ValueError("Provide a valid stock symbol.")
    return symbol


def specialized_limitations(profile):
    text = f"{profile.get('sector', '')} {profile.get('industry', '')}".lower()
    out = []
    if any(x in text for x in ["bank", "financial services", "insurance"]):
        out.append("Banks, insurers, and similar financial institutions require sector-specific leverage and cash-flow interpretation.")
    if any(x in text for x in ["reit", "real estate investment trust"]):
        out.append("REIT analysis commonly requires FFO/AFFO and property-level metrics that are not included.")
    return out


def fetch_research_data(symbol):
    today = pd.Timestamp.utcnow().normalize().tz_localize(None)
    start = today - pd.DateOffset(years=HISTORICAL_YEARS + 1)
    f, t = start.date().isoformat(), today.date().isoformat()
    profile_records = get_company_profile(symbol)
    return {
        "profile": profile_records[0] if profile_records else {},
        "income_q": get_income_statements(symbol, "quarter", 40),
        "balance_q": get_balance_sheets(symbol, "quarter", 40),
        "cash_q": get_cash_flow_statements(symbol, "quarter", 40),
        "income_a": get_income_statements(symbol, "annual", 8),
        "cash_a": get_cash_flow_statements(symbol, "annual", 8),
        "prices": get_dividend_adjusted_prices(symbol, f, t),
        "dividends": get_dividends(symbol, f, t),
        "benchmark": get_benchmark_prices(BENCHMARK_SYMBOL, f, t),
    }


def build_evidence_summary(symbol):
    symbol = normalize_symbol(symbol)
    raw = fetch_research_data(symbol)
    iq, im = normalize_statement(raw["income_q"], INCOME_ALIASES, "income_quarterly")
    bq, bm = normalize_statement(raw["balance_q"], BALANCE_ALIASES, "balance_quarterly")
    cq, cm = normalize_statement(raw["cash_q"], CASH_FLOW_ALIASES, "cash_flow_quarterly")
    ia, _ = normalize_statement(raw["income_a"], INCOME_ALIASES, "income_annual")
    ca, _ = normalize_statement(raw["cash_a"], CASH_FLOW_ALIASES, "cash_flow_annual")
    q = align_quarterly_statements(iq, bq, cq)
    prices, benchmark, dividends = normalize_prices(raw["prices"]), normalize_prices(raw["benchmark"]), normalize_dividends(raw["dividends"])
    if q.empty: raise ValueError(f"No aligned quarterly statements for {symbol}.")
    if prices.empty: raise ValueError(f"No dividend-adjusted prices for {symbol}.")
    cutoff = q["fiscal_period_end"].max() - pd.DateOffset(years=HISTORICAL_YEARS + 1)
    q = q.loc[q["fiscal_period_end"] >= cutoff].reset_index(drop=True)

    # Actual normalized date ranges used by the analysis.
    financial_statement_start_date = q["fiscal_period_end"].min()
    financial_statement_end_date = q["fiscal_period_end"].max()
    stock_price_start_date = prices["date"].min()
    stock_price_end_date = prices["date"].max()

    # Retain the existing variable names used by the metric modules.
    statement_date = financial_statement_end_date
    market_date = stock_price_end_date

    profile = raw["profile"]
    sections = {
        "growth": build_growth_metrics(q, statement_date),
        "profitability": build_profitability_metrics(q, statement_date),
        "cash_flow": build_cash_flow_metrics(q, statement_date),
        "balance_sheet": build_balance_sheet_metrics(q, statement_date),
        "capital_efficiency": build_capital_efficiency_metrics(q, statement_date),
        "capital_allocation": build_capital_allocation_metrics(q, dividends, statement_date),
        "valuation": build_valuation_metrics(profile, q, prices, market_date),
        "market_performance": build_market_metrics(prices, benchmark, market_date),
    }
    dq = build_data_quality(q, [im, bm, cm], prices, ia, ca)
    currency = q["reportedCurrency"].dropna().iloc[-1] if "reportedCurrency" in q and not q["reportedCurrency"].dropna().empty else profile.get("currency")
    limitations = [
        "Uses FMP-normalized structured data rather than primary-source filings.",
        "Excludes news, transcripts, management guidance, competitors, and macroeconomic data.",
        "Historical percentiles describe company-specific history, not peer-relative value.",
        "TTM windows may not capture fiscal-calendar changes, 53-week years, acquisitions, or divestitures.",
        "Generic ratios are not equally comparable across sectors.",
        "Market and statement dates are separate and not necessarily contemporaneous.",
        "Research assistance only; not personalized financial advice.",
    ] + specialized_limitations(profile)
    return serializable({
        "schema_version": "1.0",
        "company": {"symbol": symbol, "name": profile.get("companyName") or profile.get("name"), "exchange": profile.get("exchangeShortName") or profile.get("exchange"), "sector": profile.get("sector"), "industry": profile.get("industry"), "currency": currency},
        "as_of": {
            "analysis_generated_at": datetime.utcnow().replace(microsecond=0),
            "financial_statement_data_start_date": financial_statement_start_date,
            "financial_statement_data_end_date": financial_statement_end_date,
            "stock_price_data_start_date": stock_price_start_date,
            "stock_price_data_end_date": stock_price_end_date,
            "latest_financial_statement_date": financial_statement_end_date,
            "latest_filing_date": dq["latest_filing_date"],
            "latest_market_data_date": stock_price_end_date,
            "benchmark_symbol": BENCHMARK_SYMBOL,
        },
        "data_quality": dq, "evidence": sections, "signals": build_signals(sections), "limitations": limitations,
    })

## 11. Compact LLM payload

In [12]:
def compact_metric(item):
    allowed = {
        "current_ttm", "prior_ttm", "previous_ttm_yoy_growth", "growth_acceleration_pp",
        "historical_median_growth", "historical_median", "historical_percentile",
        "change_pp", "stock_return_1y", "benchmark_return_1y", "benchmark_symbol",
        "observations", "historical_observations",
    }
    comparison = {k: v for k, v in item.get("comparison", {}).items() if k in allowed and v is not None}
    return {"id": item["evidence_id"], "metric": item["metric"], "value": item.get("value"), "unit": item.get("unit"), "assessment": item.get("assessment"), "comparison": comparison, "notes": item.get("notes", [])}


def build_llm_payload(evidence):
    payload = {
        "schema_version": evidence["schema_version"],
        "company": evidence["company"],
        "as_of": evidence["as_of"],
        "data_quality": {
            "overall_status": evidence["data_quality"]["overall_status"],
            "number_of_aligned_quarters": evidence["data_quality"]["number_of_aligned_quarters"],
            "warnings": evidence["data_quality"]["warnings"],
            "missing_metrics": evidence["data_quality"]["missing_metrics"],
        },
        "evidence": {section: [compact_metric(item) for item in metrics.values()] for section, metrics in evidence["evidence"].items()},
        "material_signals": [
            {k: s[k] for k in ["metric_id", "assessment", "direction", "severity", "evidence_ids", "explanation"]}
            for s in evidence["signals"]
            if s["severity"] not in {"neutral", "unknown", "not_applicable"} or s["assessment"] == "conflicting_evidence"
        ],
        "limitations": evidence["limitations"],
    }
    payload = serializable(payload)
    print(f"Compact payload size: approximately {len(json.dumps(payload, separators=(',', ':'))):,} JSON characters")
    return payload

## 12. Structured OpenAI thesis output

In [13]:
class EvidenceBackedStatement(BaseModel):
    statement: str
    supporting_evidence_ids: List[str] = Field(default_factory=list)


class InvestmentThesis(BaseModel):
    company_summary: EvidenceBackedStatement
    overall_assessment: EvidenceBackedStatement
    business_quality: EvidenceBackedStatement
    operating_momentum: EvidenceBackedStatement
    financial_strength: EvidenceBackedStatement
    cash_flow_quality: EvidenceBackedStatement
    capital_allocation: EvidenceBackedStatement
    valuation_assessment: EvidenceBackedStatement
    market_assessment: EvidenceBackedStatement
    bull_case: List[EvidenceBackedStatement]
    bear_case: List[EvidenceBackedStatement]
    key_tensions: List[EvidenceBackedStatement]
    conditions_that_would_strengthen_the_thesis: List[EvidenceBackedStatement]
    conditions_that_would_weaken_the_thesis: List[EvidenceBackedStatement]
    confidence: Literal["low", "medium", "high"]
    confidence_explanation: EvidenceBackedStatement
    data_limitations: List[str]


THESIS_INSTRUCTIONS = """
Use only the supplied evidence. Do not use outside company knowledge.
Do not invent facts, events, products, management actions, guidance, competitors, catalysts, or missing data.
Do not recalculate metrics. Treat deterministic classifications as signals, not infallible conclusions.
Distinguish business quality from valuation. Acknowledge conflicting evidence.
Cite evidence IDs for every material conclusion. State when evidence is insufficient.
Do not give personalized financial advice or issue a simplistic buy/sell command.
Conditions must be measurable extensions of supplied evidence, not invented catalysts.
Produce a balanced, evidence-grounded investment thesis.
"""


def generate_structured_thesis(payload):
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.responses.parse(
        model=OPENAI_MODEL,
        instructions=THESIS_INSTRUCTIONS,
        input=json.dumps(payload, ensure_ascii=False),
        text_format=InvestmentThesis,
        reasoning={"effort": "low"},
        max_output_tokens=5000,
    )
    if response.output_parsed is None:
        raise ValueError("No parsed thesis was returned.")
    return response.output_parsed, response

## 13. Result object and notebook display

In [14]:
def format_statement(item):
    ids = ", ".join(item.supporting_evidence_ids) or "none"
    return f"{item.statement}  \n*Evidence: {ids}*"


def format_list(items):
    return "\n".join(f"- {x.statement} **[{', '.join(x.supporting_evidence_ids) or 'no evidence ID'}]**" for x in items) or "- No supported conclusion."


@dataclass
class EquityResearchResult:
    symbol: str
    financial_statement_start_date: Optional[str]
    financial_statement_end_date: Optional[str]
    stock_price_start_date: Optional[str]
    stock_price_end_date: Optional[str]
    evidence: dict
    llm_payload: dict
    thesis: InvestmentThesis
    model: str
    raw_response: object

    def _repr_markdown_(self):
        c, a, t = self.evidence["company"], self.evidence["as_of"], self.thesis
        return f"""
# {c.get('name') or self.symbol} ({self.symbol})

**Financial statement data:** {self.financial_statement_start_date} through {self.financial_statement_end_date}
**Stock price data:** {self.stock_price_start_date} through {self.stock_price_end_date}
**Latest filing date:** {a.get('latest_filing_date')}
**Model:** `{self.model}`

## Overall assessment
{format_statement(t.overall_assessment)}

## Business quality
{format_statement(t.business_quality)}

## Operating momentum
{format_statement(t.operating_momentum)}

## Financial strength
{format_statement(t.financial_strength)}

## Cash-flow quality
{format_statement(t.cash_flow_quality)}

## Capital allocation
{format_statement(t.capital_allocation)}

## Valuation
{format_statement(t.valuation_assessment)}

## Market behavior
{format_statement(t.market_assessment)}

## Bull case
{format_list(t.bull_case)}

## Bear case
{format_list(t.bear_case)}

## Key tensions
{format_list(t.key_tensions)}

## Conditions that strengthen the thesis
{format_list(t.conditions_that_would_strengthen_the_thesis)}

## Conditions that weaken the thesis
{format_list(t.conditions_that_would_weaken_the_thesis)}

## Confidence: {t.confidence.title()}
{format_statement(t.confidence_explanation)}

## Data limitations
{chr(10).join('- ' + x for x in t.data_limitations)}
"""

    def display(self):
        display(Markdown(self._repr_markdown_()))

    def save_evidence_json(self, path=None):
        path = Path(path or f"{self.symbol}_canonical_evidence.json")
        path.write_text(json.dumps(serializable(self.evidence), indent=2), encoding="utf-8")
        return path

    def save_llm_payload_json(self, path=None):
        path = Path(path or f"{self.symbol}_llm_payload.json")
        path.write_text(json.dumps(serializable(self.llm_payload), indent=2), encoding="utf-8")
        return path

    def save_thesis_json(self, path=None):
        path = Path(path or f"{self.symbol}_structured_thesis.json")
        path.write_text(self.thesis.model_dump_json(indent=2), encoding="utf-8")
        return path

## 14. Public one-argument function

In [15]:
def analyze_stock(symbol):
    normalized = normalize_symbol(symbol)
    evidence = build_evidence_summary(normalized)
    payload = build_llm_payload(evidence)
    thesis, raw_response = generate_structured_thesis(payload)

    as_of = evidence.get("as_of", {})

    return EquityResearchResult(
        symbol=normalized,
        financial_statement_start_date=as_of.get(
            "financial_statement_data_start_date"
        ),
        financial_statement_end_date=as_of.get(
            "financial_statement_data_end_date"
        ),
        stock_price_start_date=as_of.get(
            "stock_price_data_start_date"
        ),
        stock_price_end_date=as_of.get(
            "stock_price_data_end_date"
        ),
        evidence=evidence,
        llm_payload=payload,
        thesis=thesis,
        model=OPENAI_MODEL,
        raw_response=raw_response,
    )


## 15. Demonstration

In [17]:
result = analyze_stock("NVDA")
result

/tmp/ipykernel_3273/4261096341.py:87: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "analysis_generated_at": datetime.utcnow().replace(microsecond=0),


Compact payload size: approximately 17,311 JSON characters



# NVIDIA Corporation (NVDA)

**Financial statement data:** 2016-07-31T00:00:00 through 2026-04-26T00:00:00
**Stock price data:** 2015-06-18T00:00:00 through 2026-06-18T00:00:00
**Latest filing date:** 2026-05-20T00:00:00
**Model:** `gpt-5.4-mini`

## Overall assessment
The evidence supports a high-quality business with exceptional current operating momentum and financial strength, but valuation is not obviously cheap despite being below some of NVIDIA’s own historical medians; the main tension is whether the current growth and margin profile can persist enough to justify the stock’s still-elevated cash-flow and earnings multiples.  
*Evidence: G01, P01, P02, P03, BS03, BS04, BS05, BS06, V01, V03, V04, V05, V06*

## Business quality
Business quality appears very strong: gross margin is 74.1%, operating margin is 64.0%, and net margin is 63.0%, each above or strongly above historical levels, while ROA, ROE, and ROIC are all high and classified as in line with history or better.  
*Evidence: P01, P02, P03, CE01, CE02, CE03*

## Operating momentum
Operating momentum is robust, with revenue, gross profit, EPS, and net income growth all well above historical medians or near the upper part of the company’s history; cash flow growth is also solid, though less extreme than earnings growth.  
*Evidence: G01, G02, G04, G05, G06, G07*

## Financial strength
Financial strength is excellent: current ratio is 3.44, interest coverage is 544.6x, net debt to EBITDA is negative, and cash growth outpaced debt growth, though debt itself still rose 24.6% year over year.  
*Evidence: BS04, BS05, BS06, BS07, BS08, BS02, BS01*

## Cash-flow quality
Cash flow quality looks healthy but not perfect. Operating cash flow is 78.7% of net income and free cash flow is 74.6% of net income, indicating strong conversion, while free cash flow margin remains high at 47.0%.  
*Evidence: CF01, CF02, P05, P04*

## Capital allocation
Capital allocation is shareholder-return oriented, with $47.4B of share repurchases and $973M of dividends, but repurchases and total capital returned consumed only about 40% of free cash flow, and diluted share count still declined only modestly.  
*Evidence: CA01, CA02, CA05, CA06, CA04*

## Valuation
Valuation looks mixed rather than clearly cheap. P/E of 31.7x, EV/EBITDA of 25.9x, and price-to-free-cash-flow of 42.5x are all below the company’s historical medians and classified above_history, but they still represent substantial multiples in absolute terms.  
*Evidence: V01, V03, V04, V05, V06, V02*

## Market behavior
The stock has performed strongly over longer periods, with 1-year return of 43.5% and 3-year annualized return of 69.8%, outperforming SPY over the past year, but it also experienced a large maximum drawdown of -66.3% and still trades 11.5% below its 52-week high.  
*Evidence: M04, M05, M09, M07, M08, M06, M10*

## Bull case
- If the current TTM trend continues, NVIDIA’s very high revenue and profit growth could support further earnings expansion from an already elevated base. **[G01, G02, G04, G05]**
- The company’s margin profile is exceptional relative to its own history, which supports the view that the current business mix is highly profitable. **[P01, P02, P03]**
- The balance sheet provides meaningful cushion, with large cash holdings, low debt, negative net debt, and very strong interest coverage. **[BS01, BS02, BS03, BS05, BS06]**
- Share repurchases and dividends indicate management is returning cash to shareholders while still preserving a strong net cash position. **[CA01, CA02, CA06, BS03]**

## Bear case
- Valuation remains demanding in absolute terms, especially on earnings and free cash flow multiples, which leaves less room for disappointment. **[V01, V04, V03, V06]**
- Debt increased year over year even though the company remains net cash positive, so balance-sheet strength is good but not static. **[BS07, BS02, BS03]**
- The business has shown large price volatility and historical drawdowns, suggesting the market may continue to assign a high-risk profile to the shares. **[M06, M07, M10]**
- Cash flow conversion is solid but below 1.0x net income, implying reported earnings are not fully matched by operating and free cash flow. **[CF01, CF02]**

## Key tensions
- Growth and margin quality are outstanding, but valuation is only attractive relative to NVIDIA’s own history rather than obviously inexpensive on an absolute basis. **[G01, P02, P03, V01, V03, V04]**
- The company is producing large profits and cash, yet shareholder returns are meaningful but not large enough to materially reduce share count quickly. **[CA02, CA05, CA06, CA04]**
- The balance sheet is very strong, but the stock remains volatile and has suffered large historical drawdowns. **[BS03, BS06, M07, M10]**

## Conditions that strengthen the thesis
- Evidence that revenue, gross profit, operating income, and diluted EPS continue growing at rates near or above the current TTM levels in subsequent aligned quarters would strengthen the growth thesis. **[G01, G02, G03, G04, G05]**
- Sustained gross margin, operating margin, and net margin remaining near current highs would strengthen the quality thesis. **[P01, P02, P03]**
- Continuation of strong cash conversion, with operating cash flow and free cash flow remaining a high fraction of net income, would strengthen confidence in earnings quality. **[CF01, CF02, P04, P05]**
- If debt growth stays below cash growth and net debt remains negative while interest coverage stays extremely high, the balance-sheet thesis would be reinforced. **[BS03, BS06, BS07, BS08]**

## Conditions that weaken the thesis
- A clear slowdown in revenue or gross profit growth toward or below the company’s historical medians would weaken the current momentum thesis. **[G01, G02]**
- Compression in operating margin or net margin back toward historical norms would weaken the high-quality profitability view. **[P02, P03]**
- A material deterioration in cash flow conversion, such as operating cash flow or free cash flow falling well below net income, would weaken cash quality. **[CF01, CF02]**
- If debt continues rising faster than cash and leverage metrics move away from net cash territory, the balance-sheet case would weaken. **[BS02, BS03, BS07, BS08]**
- If valuation multiples expand materially from current levels without a corresponding improvement in growth or margins, the investment case would weaken. **[V01, V03, V04]**

## Confidence: High
Confidence is high because the dataset is high quality, aligned across 40 quarters, and the key conclusions are supported by multiple consistent signals across growth, margins, cash flow, balance sheet, and valuation.  
*Evidence: data_quality, G01, P02, P03, BS03, BS06, V01, M09*

## Data limitations
- Uses FMP-normalized structured data rather than primary-source filings.
- Excludes news, transcripts, management guidance, competitors, and macroeconomic data.
- Historical percentiles describe company-specific history, not peer-relative value.
- TTM windows may not capture fiscal-calendar changes, 53-week years, acquisitions, or divestitures.
- Generic ratios are not equally comparable across sectors.
- Market and statement dates are separate and not necessarily contemporaneous.
- Research assistance only; not personalized financial advice.


In [18]:
canonical_preview = {
    "company": result.evidence["company"],
    "as_of": result.evidence["as_of"],
    "data_quality": result.evidence["data_quality"],
    "selected_evidence": {
        "revenue_growth": result.evidence["evidence"]["growth"]["revenue_growth"],
        "operating_margin": result.evidence["evidence"]["profitability"]["operating_margin"],
        "free_cash_flow_yield": result.evidence["evidence"]["valuation"]["free_cash_flow_yield"],
        "relative_return_1y": result.evidence["evidence"]["market_performance"]["relative_return_1y"],
    },
}
display(canonical_preview)

{'company': {'symbol': 'NVDA',
  'name': 'NVIDIA Corporation',
  'exchange': 'NASDAQ',
  'sector': 'Technology',
  'industry': 'Semiconductors',
  'currency': 'USD'},
 'as_of': {'analysis_generated_at': '2026-06-18T14:19:57',
  'financial_statement_data_start_date': '2016-07-31T00:00:00',
  'financial_statement_data_end_date': '2026-04-26T00:00:00',
  'stock_price_data_start_date': '2015-06-18T00:00:00',
  'stock_price_data_end_date': '2026-06-18T00:00:00',
  'latest_financial_statement_date': '2026-04-26T00:00:00',
  'latest_filing_date': '2026-05-20T00:00:00',
  'latest_market_data_date': '2026-06-18T00:00:00',
  'benchmark_symbol': 'SPY'},
 'data_quality': {'overall_status': 'good',
  'number_of_aligned_quarters': 40,
  'latest_filing_date': '2026-05-20T00:00:00',
  'latest_market_date': '2026-06-18T00:00:00',
  'latest_periods_consecutive': True,
  'duplicate_periods_resolved': 0,
  'annual_reconciliation': {'revenue': {'status': 'insufficient_matching_quarters',
    'fiscal_year':

In [19]:
print(json.dumps(result.llm_payload, indent=2, ensure_ascii=False))

{
  "schema_version": "1.0",
  "company": {
    "symbol": "NVDA",
    "name": "NVIDIA Corporation",
    "exchange": "NASDAQ",
    "sector": "Technology",
    "industry": "Semiconductors",
    "currency": "USD"
  },
  "as_of": {
    "analysis_generated_at": "2026-06-18T14:19:57",
    "financial_statement_data_start_date": "2016-07-31T00:00:00",
    "financial_statement_data_end_date": "2026-04-26T00:00:00",
    "stock_price_data_start_date": "2015-06-18T00:00:00",
    "stock_price_data_end_date": "2026-06-18T00:00:00",
    "latest_financial_statement_date": "2026-04-26T00:00:00",
    "latest_filing_date": "2026-05-20T00:00:00",
    "latest_market_data_date": "2026-06-18T00:00:00",
    "benchmark_symbol": "SPY"
  },
  "data_quality": {
    "overall_status": "good",
    "number_of_aligned_quarters": 40,
    "warnings": [],
    "missing_metrics": []
  },
  "evidence": {
    "growth": [
      {
        "id": "G01",
        "metric": "revenue_growth",
        "value": 0.7068376931623068,
   

In [20]:
result.display()


# NVIDIA Corporation (NVDA)

**Financial statement data:** 2016-07-31T00:00:00 through 2026-04-26T00:00:00
**Stock price data:** 2015-06-18T00:00:00 through 2026-06-18T00:00:00
**Latest filing date:** 2026-05-20T00:00:00
**Model:** `gpt-5.4-mini`

## Overall assessment
The evidence supports a high-quality business with exceptional current operating momentum and financial strength, but valuation is not obviously cheap despite being below some of NVIDIA’s own historical medians; the main tension is whether the current growth and margin profile can persist enough to justify the stock’s still-elevated cash-flow and earnings multiples.  
*Evidence: G01, P01, P02, P03, BS03, BS04, BS05, BS06, V01, V03, V04, V05, V06*

## Business quality
Business quality appears very strong: gross margin is 74.1%, operating margin is 64.0%, and net margin is 63.0%, each above or strongly above historical levels, while ROA, ROE, and ROIC are all high and classified as in line with history or better.  
*Evidence: P01, P02, P03, CE01, CE02, CE03*

## Operating momentum
Operating momentum is robust, with revenue, gross profit, EPS, and net income growth all well above historical medians or near the upper part of the company’s history; cash flow growth is also solid, though less extreme than earnings growth.  
*Evidence: G01, G02, G04, G05, G06, G07*

## Financial strength
Financial strength is excellent: current ratio is 3.44, interest coverage is 544.6x, net debt to EBITDA is negative, and cash growth outpaced debt growth, though debt itself still rose 24.6% year over year.  
*Evidence: BS04, BS05, BS06, BS07, BS08, BS02, BS01*

## Cash-flow quality
Cash flow quality looks healthy but not perfect. Operating cash flow is 78.7% of net income and free cash flow is 74.6% of net income, indicating strong conversion, while free cash flow margin remains high at 47.0%.  
*Evidence: CF01, CF02, P05, P04*

## Capital allocation
Capital allocation is shareholder-return oriented, with $47.4B of share repurchases and $973M of dividends, but repurchases and total capital returned consumed only about 40% of free cash flow, and diluted share count still declined only modestly.  
*Evidence: CA01, CA02, CA05, CA06, CA04*

## Valuation
Valuation looks mixed rather than clearly cheap. P/E of 31.7x, EV/EBITDA of 25.9x, and price-to-free-cash-flow of 42.5x are all below the company’s historical medians and classified above_history, but they still represent substantial multiples in absolute terms.  
*Evidence: V01, V03, V04, V05, V06, V02*

## Market behavior
The stock has performed strongly over longer periods, with 1-year return of 43.5% and 3-year annualized return of 69.8%, outperforming SPY over the past year, but it also experienced a large maximum drawdown of -66.3% and still trades 11.5% below its 52-week high.  
*Evidence: M04, M05, M09, M07, M08, M06, M10*

## Bull case
- If the current TTM trend continues, NVIDIA’s very high revenue and profit growth could support further earnings expansion from an already elevated base. **[G01, G02, G04, G05]**
- The company’s margin profile is exceptional relative to its own history, which supports the view that the current business mix is highly profitable. **[P01, P02, P03]**
- The balance sheet provides meaningful cushion, with large cash holdings, low debt, negative net debt, and very strong interest coverage. **[BS01, BS02, BS03, BS05, BS06]**
- Share repurchases and dividends indicate management is returning cash to shareholders while still preserving a strong net cash position. **[CA01, CA02, CA06, BS03]**

## Bear case
- Valuation remains demanding in absolute terms, especially on earnings and free cash flow multiples, which leaves less room for disappointment. **[V01, V04, V03, V06]**
- Debt increased year over year even though the company remains net cash positive, so balance-sheet strength is good but not static. **[BS07, BS02, BS03]**
- The business has shown large price volatility and historical drawdowns, suggesting the market may continue to assign a high-risk profile to the shares. **[M06, M07, M10]**
- Cash flow conversion is solid but below 1.0x net income, implying reported earnings are not fully matched by operating and free cash flow. **[CF01, CF02]**

## Key tensions
- Growth and margin quality are outstanding, but valuation is only attractive relative to NVIDIA’s own history rather than obviously inexpensive on an absolute basis. **[G01, P02, P03, V01, V03, V04]**
- The company is producing large profits and cash, yet shareholder returns are meaningful but not large enough to materially reduce share count quickly. **[CA02, CA05, CA06, CA04]**
- The balance sheet is very strong, but the stock remains volatile and has suffered large historical drawdowns. **[BS03, BS06, M07, M10]**

## Conditions that strengthen the thesis
- Evidence that revenue, gross profit, operating income, and diluted EPS continue growing at rates near or above the current TTM levels in subsequent aligned quarters would strengthen the growth thesis. **[G01, G02, G03, G04, G05]**
- Sustained gross margin, operating margin, and net margin remaining near current highs would strengthen the quality thesis. **[P01, P02, P03]**
- Continuation of strong cash conversion, with operating cash flow and free cash flow remaining a high fraction of net income, would strengthen confidence in earnings quality. **[CF01, CF02, P04, P05]**
- If debt growth stays below cash growth and net debt remains negative while interest coverage stays extremely high, the balance-sheet thesis would be reinforced. **[BS03, BS06, BS07, BS08]**

## Conditions that weaken the thesis
- A clear slowdown in revenue or gross profit growth toward or below the company’s historical medians would weaken the current momentum thesis. **[G01, G02]**
- Compression in operating margin or net margin back toward historical norms would weaken the high-quality profitability view. **[P02, P03]**
- A material deterioration in cash flow conversion, such as operating cash flow or free cash flow falling well below net income, would weaken cash quality. **[CF01, CF02]**
- If debt continues rising faster than cash and leverage metrics move away from net cash territory, the balance-sheet case would weaken. **[BS02, BS03, BS07, BS08]**
- If valuation multiples expand materially from current levels without a corresponding improvement in growth or margins, the investment case would weaken. **[V01, V03, V04]**

## Confidence: High
Confidence is high because the dataset is high quality, aligned across 40 quarters, and the key conclusions are supported by multiple consistent signals across growth, margins, cash flow, balance sheet, and valuation.  
*Evidence: data_quality, G01, P02, P03, BS03, BS06, V01, M09*

## Data limitations
- Uses FMP-normalized structured data rather than primary-source filings.
- Excludes news, transcripts, management guidance, competitors, and macroeconomic data.
- Historical percentiles describe company-specific history, not peer-relative value.
- TTM windows may not capture fiscal-calendar changes, 53-week years, acquisitions, or divestitures.
- Generic ratios are not equally comparable across sectors.
- Market and statement dates are separate and not necessarily contemporaneous.
- Research assistance only; not personalized financial advice.


In [21]:
print("Symbol:", result.symbol)
print("Model:", result.model)

print(
    "Financial statement data range:",
    result.financial_statement_start_date,
    "through",
    result.financial_statement_end_date,
)

print(
    "Stock price data range:",
    result.stock_price_start_date,
    "through",
    result.stock_price_end_date,
)

display(result.evidence["evidence"]["growth"]["revenue_growth"])
display(result.thesis.overall_assessment.model_dump())


Symbol: NVDA
Model: gpt-5.4-mini
Financial statement data range: 2016-07-31T00:00:00 through 2026-04-26T00:00:00
Stock price data range: 2015-06-18T00:00:00 through 2026-06-18T00:00:00


{'evidence_id': 'G01',
 'metric': 'revenue_growth',
 'value': 0.7068376931623068,
 'unit': 'ratio',
 'as_of': '2026-04-26T00:00:00',
 'comparison': {'current_ttm': 253491000000.0,
  'prior_ttm': 148515000000.0,
  'previous_ttm_yoy_growth': 0.6547353579009478,
  'growth_acceleration_pp': 5.210233526135899,
  'historical_median_growth': 0.5272943762593882,
  'historical_q25_growth': 0.17681469885474166,
  'historical_q75_growth': 0.6760045924225029,
  'historical_percentile': 0.7878787878787878,
  'observations': 33},
 'assessment': 'above_history',
 'formula': 'current TTM / prior TTM - 1',
 'notes': []}

{'statement': 'The evidence supports a high-quality business with exceptional current operating momentum and financial strength, but valuation is not obviously cheap despite being below some of NVIDIA’s own historical medians; the main tension is whether the current growth and margin profile can persist enough to justify the stock’s still-elevated cash-flow and earnings multiples.',
 'supporting_evidence_ids': ['G01',
  'P01',
  'P02',
  'P03',
  'BS03',
  'BS04',
  'BS05',
  'BS06',
  'V01',
  'V03',
  'V04',
  'V05',
  'V06']}

In [22]:
print(result.save_evidence_json())
print(result.save_llm_payload_json())
print(result.save_thesis_json())

NVDA_canonical_evidence.json
NVDA_llm_payload.json
NVDA_structured_thesis.json


## 16. Methodology and limitations

Deterministic transformations improve auditability because each metric has an explicit source, formula, date, unit, comparison basis, and evidence ID. The LLM is used only for synthesis and receives a compact derived payload rather than raw statements.

This approach can replace RAG for some structured-data questions, but it does not replace retrieval over filings, transcripts, news, guidance, footnotes, or qualitative competitive research.

Important limitations:

- FMP-normalized fields can differ from primary filings or change after restatements.
- Generic ratios are not equally meaningful across sectors.
- Banks, insurers, REITs, and other specialized structures need sector-specific metrics.
- TTM windows can obscure fiscal-calendar changes, 53-week years, seasonality, acquisitions, and divestitures.
- Historical company percentiles are not peer-relative valuation benchmarks.
- Company quality and investment attractiveness are distinct; valuation can offset strong fundamentals.
- Financial-statement and market-data dates remain separate.
- The output is research assistance, not financial advice.